# Experiment: frozen NVARC on an ARC-AGI-1 public-evaluation subset (pass@32)

**Question:** How often does the released NVARC/Qwen3-4B checkpoint solve the 396 retained public ARC-AGI-1 evaluation tasks with up to 32 direct stochastic attempts and no test-time training?

This is a deliberately narrow frozen-model baseline. It uses canonical prompts and excludes test-time training, test-time augmentation, DFS, beam search, candidate rescoring, voting, invalid-output retries, and target-shape hints. Four named tasks whose complete prompts cannot fit the established 8,192-token protocol are excluded rather than truncated. The checkpoint is known to be contaminated with public-evaluation signal, so the result is a subset checkpoint-behavior measurement—not a generalization estimate.

The headline metric is **strict task pass@k**: for a task with multiple test inputs, one common attempt index must solve every test output. Output-level and task-averaged metrics are reported only as diagnostics.

## Runtime requirements

1. Start a fresh Google Colab runtime with one BF16-capable NVIDIA GPU and at least 20 GiB VRAM (L4, A100, H100, or newer). A T4 is intentionally rejected.
2. Confirm that `MyDrive/diffusion-arc/models/nvarc-qwen3-4b-grids15-sft139` exists.
3. Run the notebook from top to bottom. The first two prompts form a persisted smoke batch; the full run then resumes from it.

The model is copied from Drive to the Colab SSD for faster random reads. Per-output results remain on Drive and are safe to resume after a runtime interruption.

## 1. Install the pinned inference runtime

Run this before importing Torch. vLLM installs its compatible CUDA/PyTorch stack. Installation output is hidden on success and shown on failure.

In [ ]:
%pip uninstall -y torchvision torchaudio

In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

VLLM_VERSION = "0.27.1"

try:
    installed_vllm = importlib.metadata.version("vllm")
except importlib.metadata.PackageNotFoundError:
    installed_vllm = None

if installed_vllm != VLLM_VERSION:
    completed = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"vllm=={VLLM_VERSION}"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if completed.returncode != 0:
        print(completed.stdout[-12000:])
        completed.check_returncode()

assert importlib.metadata.version("vllm") == VLLM_VERSION
print(f"vLLM {VLLM_VERSION} is installed.")

## 2. Mount Drive and lock the experiment configuration

The deterministic run name is intentional: rerunning the notebook resumes matching result files and rejects files produced by a different configuration.

In [ ]:
import json
import os
import platform
import random
import shutil
import time
from pathlib import Path

import torch
from google.colab import drive

NOTEBOOK_PROCEDURE_VERSION = 1
SEED = 42
SEED_NAMESPACE = "nvarc-arc-agi-1-pass32-v1"
SAMPLES_PER_PROMPT = 32
PASS_AT_K = (1, 2, 4, 8, 16, 32)
TEMPERATURE = 1.0
TOP_P = 1.0
TOP_K = -1
MAX_MODEL_LEN = 8192
PROMPT_BATCH_SIZE = 8
SMOKE_PROMPTS = 2
EVALUATION_TASK_LIMIT = None  # Keep None for all 396 retained tasks.
EXCLUDED_CONTEXT_TASK_IDS = ("47996f11", "981571dc", "af22c60d", "f9d67f8b")
GPU_MEMORY_UTILIZATION = 0.90

ARC_TOKEN_IDS = tuple(range(11)) + (15,)
EOS_TOKEN_ID = 15
DATASET_REPOSITORY = "https://github.com/fchollet/ARC-AGI"
DATASET_RELEASE = "v1.0.2"
DATASET_ARCHIVE_URL = (
    "https://codeload.github.com/fchollet/ARC-AGI/tar.gz/refs/tags/v1.0.2"
)
EXPECTED_DATASET_ARCHIVE_SHA256 = (
    "159676f5c7c6e3b74e89403edda319ff2be714b630e9594ac64914ba1e788466"
)
NVARC_HANDLE = "sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
EXPECTED_NVARC_FILE_SIZES = {
    "model-00001-of-00002.safetensors": 4_996_836_472,
    "model-00002-of-00002.safetensors": 2_270_397_024,
}

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "diffusion-arc"
MODEL_DRIVE_DIR = DRIVE_ROOT / "models" / "nvarc-qwen3-4b-grids15-sft139"
MODEL_LOCAL_DIR = Path("/content/checkpoints/nvarc-qwen3-4b-grids15-sft139")
RUN_NAME = "arc-agi-1-v1.0.2-subset396-s32-temp1-seed42-v1"
RUN_DIR = DRIVE_ROOT / "results" / "nvarc-arc-agi-1-pass32" / RUN_NAME
RESULT_DIR = RUN_DIR / "output_results"
BATCH_EVENT_DIR = RUN_DIR / "batch_events"
SESSION_DIR = RUN_DIR / "sessions"

assert SAMPLES_PER_PROMPT == 32
assert PASS_AT_K[-1] == SAMPLES_PER_PROMPT
assert TEMPERATURE > 0 and TOP_P == 1.0 and TOP_K == -1
assert EVALUATION_TASK_LIMIT is None or EVALUATION_TASK_LIMIT > 0
assert len(EXCLUDED_CONTEXT_TASK_IDS) == len(set(EXCLUDED_CONTEXT_TASK_IDS)) == 4
assert set(ARC_TOKEN_IDS) == set(range(11)) | {EOS_TOKEN_ID}

drive.mount(str(DRIVE_MOUNT), force_remount=False)
assert MODEL_DRIVE_DIR.is_dir(), MODEL_DRIVE_DIR
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_EVENT_DIR.mkdir(parents=True, exist_ok=True)
SESSION_DIR.mkdir(parents=True, exist_ok=True)
SESSION_ID = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + f"-{time.time_ns() % 10**9:09d}"

assert torch.cuda.is_available(), "CUDA is required"
assert torch.cuda.is_bf16_supported(), (
    "This experiment requires native BF16 support; choose L4/A100/H100 or newer."
)
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 20.0, (gpu.name, gpu_memory_gib)

random.seed(SEED)
torch.manual_seed(SEED)

runtime_preflight = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "vllm": importlib.metadata.version("vllm"),
    "transformers": importlib.metadata.version("transformers"),
    "safetensors": importlib.metadata.version("safetensors"),
    "gpu": gpu.name,
    "gpu_memory_GiB": round(gpu_memory_gib, 2),
    "gpu_count": torch.cuda.device_count(),
    "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
    "bf16_supported": torch.cuda.is_bf16_supported(),
}
runtime_preflight

## 3. Pure helpers and self-tests

These functions own stable seeds, exact ARC serialization, strict parsing, hashing, and atomic JSON persistence. They have no model or label dependency.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from typing import Any


def stable_seed(*parts: object) -> int:
    payload = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big") % (2**31)


def is_valid_grid(value: object) -> bool:
    if not isinstance(value, list) or not value or len(value) > 30:
        return False
    if any(not isinstance(row, list) or not row or len(row) > 30 for row in value):
        return False
    width = len(value[0])
    if any(len(row) != width for row in value):
        return False
    return all(
        isinstance(cell, int) and not isinstance(cell, bool) and 0 <= cell <= 9
        for row in value
        for cell in row
    )


def copy_grid(grid: list[list[int]]) -> list[list[int]]:
    assert is_valid_grid(grid)
    return [[int(cell) for cell in row] for row in grid]


def serialize_grid(grid: list[list[int]]) -> str:
    assert is_valid_grid(grid)
    return "\n".join("".join(str(cell) for cell in row) for row in grid)


def parse_grid_text(text: str) -> list[list[int]] | None:
    lines = text.strip().splitlines()
    if not lines or len(lines) > 30:
        return None
    rows: list[list[int]] = []
    for raw_line in lines:
        line = raw_line.strip()
        if not line or len(line) > 30 or any(character not in "0123456789" for character in line):
            return None
        rows.append([int(character) for character in line])
    return rows if is_valid_grid(rows) else None


def format_nvarc_prompt(task: dict[str, Any], test_index: int) -> str:
    parts: list[str] = []
    for example in task["train"]:
        parts.extend(
            (
                f"<|im_start|>user\n{serialize_grid(example['input'])}<|im_end|>",
                f"<|im_start|>assistant\n{serialize_grid(example['output'])}<|im_end|>",
            )
        )
    test_input = task["test"][test_index]["input"]
    parts.extend(
        (
            f"<|im_start|>user\n{serialize_grid(test_input)}<|im_end|>",
            "<|im_start|>assistant\n",
        )
    )
    return "".join(parts)


def exact_grid_match(left: object, right: object) -> bool:
    return is_valid_grid(left) and is_valid_grid(right) and left == right


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_json_sha256(value: object) -> str:
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def atomic_write_json(path: str | Path, value: object, *, indent: int | None = 2) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=indent), encoding="utf-8")
    os.replace(temporary, destination)


# Local, model-free invariants.
assert stable_seed("x", 1) == stable_seed("x", 1)
assert serialize_grid([[0, 1], [2, 3]]) == "01\n23"
assert parse_grid_text("01\n23") == [[0, 1], [2, 3]]
assert parse_grid_text("01\n2") is None
assert parse_grid_text("0a") is None
assert not is_valid_grid([[True]])
_tiny_prompt = format_nvarc_prompt(
    {
        "train": [{"input": [[1]], "output": [[2]]}],
        "test": [{"input": [[3]]}],
    },
    0,
)
assert _tiny_prompt == (
    "<|im_start|>user\n1<|im_end|>"
    "<|im_start|>assistant\n2<|im_end|>"
    "<|im_start|>user\n3<|im_end|>"
    "<|im_start|>assistant\n"
)
print("Pure helper self-tests passed.")

## 4. Download and validate ARC-AGI-1 v1.0.2

The official release archive is hash-pinned and stored only on the ephemeral Colab disk. All 400 evaluation files are validated before the four approved context exclusions are applied. Test outputs are copied into a separate scoring object; the model-facing challenge object retains only test inputs.

In [ ]:
import tarfile
import urllib.request

DATASET_PARENT = Path("/content/datasets/arc-agi-1-v1.0.2")
DATASET_ARCHIVE = DATASET_PARENT / "arc-agi-1-v1.0.2.tar.gz"
DATASET_PARENT.mkdir(parents=True, exist_ok=True)

if not DATASET_ARCHIVE.is_file():
    request = urllib.request.Request(
        DATASET_ARCHIVE_URL, headers={"User-Agent": "diffusion-arc-colab/1"}
    )
    temporary_archive = DATASET_ARCHIVE.with_suffix(".tmp")
    with urllib.request.urlopen(request, timeout=120) as response, temporary_archive.open("wb") as output:
        shutil.copyfileobj(response, output)
    os.replace(temporary_archive, DATASET_ARCHIVE)

dataset_archive_sha256 = sha256_file(DATASET_ARCHIVE)
assert dataset_archive_sha256 == EXPECTED_DATASET_ARCHIVE_SHA256, dataset_archive_sha256

evaluation_candidates = sorted(DATASET_PARENT.glob("*/data/evaluation"))
if not evaluation_candidates:
    extraction_root = DATASET_PARENT.resolve()
    with tarfile.open(DATASET_ARCHIVE, "r:gz") as archive:
        members = archive.getmembers()
        for member in members:
            target = (DATASET_PARENT / member.name).resolve()
            if not target.is_relative_to(extraction_root):
                raise RuntimeError(f"Unsafe archive member: {member.name}")
            if not (member.isfile() or member.isdir()):
                raise RuntimeError(f"Unsupported archive member: {member.name}")
        archive.extractall(DATASET_PARENT, members=members, filter="data")
    evaluation_candidates = sorted(DATASET_PARENT.glob("*/data/evaluation"))

assert len(evaluation_candidates) == 1, evaluation_candidates
EVALUATION_DIR = evaluation_candidates[0]
evaluation_files = sorted(EVALUATION_DIR.glob("*.json"))
assert len(evaluation_files) == 400, len(evaluation_files)

evaluation_challenges: dict[str, dict] = {}
evaluation_targets: dict[str, list[list[list[int]]]] = {}
task_file_hashes: dict[str, str] = {}

for task_path in evaluation_files:
    task_id = task_path.stem
    payload = json.loads(task_path.read_text(encoding="utf-8"))
    unexpected_task_fields = set(payload) - {"train", "test", "name"}
    assert not unexpected_task_fields, (task_id, unexpected_task_fields)
    assert {"train", "test"}.issubset(payload), (task_id, payload.keys())
    if "name" in payload:
        assert isinstance(payload["name"], str) and payload["name"]
    assert isinstance(payload["train"], list) and payload["train"]
    assert isinstance(payload["test"], list) and payload["test"]
    train = []
    for example in payload["train"]:
        assert set(example) == {"input", "output"}
        assert is_valid_grid(example["input"]) and is_valid_grid(example["output"])
        train.append({"input": copy_grid(example["input"]), "output": copy_grid(example["output"])})
    tests = []
    targets = []
    for example in payload["test"]:
        assert set(example) == {"input", "output"}
        assert is_valid_grid(example["input"]) and is_valid_grid(example["output"])
        tests.append({"input": copy_grid(example["input"])})
        targets.append(copy_grid(example["output"]))
    evaluation_challenges[task_id] = {"train": train, "test": tests}
    evaluation_targets[task_id] = targets
    task_file_hashes[task_path.name] = sha256_file(task_path)

all_task_ids = sorted(evaluation_challenges)
assert set(EXCLUDED_CONTEXT_TASK_IDS).issubset(all_task_ids)
eligible_task_ids = [
    task_id for task_id in all_task_ids if task_id not in EXCLUDED_CONTEXT_TASK_IDS
]
selected_task_ids = (
    eligible_task_ids
    if EVALUATION_TASK_LIMIT is None
    else eligible_task_ids[:EVALUATION_TASK_LIMIT]
)
total_output_count = sum(len(evaluation_targets[task_id]) for task_id in all_task_ids)
selected_output_count = sum(
    len(evaluation_challenges[task_id]["test"]) for task_id in selected_task_ids
)
assert len(all_task_ids) == 400 and total_output_count == 419
if EVALUATION_TASK_LIMIT is None:
    assert len(selected_task_ids) == 396 and selected_output_count == 415
dataset_manifest = {
    "repository": DATASET_REPOSITORY,
    "release": DATASET_RELEASE,
    "archive_url": DATASET_ARCHIVE_URL,
    "archive_sha256": dataset_archive_sha256,
    "evaluation_task_count": len(all_task_ids),
    "evaluation_output_count": total_output_count,
    "excluded_task_ids": list(EXCLUDED_CONTEXT_TASK_IDS),
    "exclusion_reason": (
        "Complete canonical prompt plus maximum shape-agnostic output has a proven "
        "minimum of 9,292 tokens, exceeding the established 8,192-token protocol."
    ),
    "selected_task_count": len(selected_task_ids),
    "selected_output_count": selected_output_count,
    "task_files_sha256": canonical_json_sha256(task_file_hashes),
}
atomic_write_json(RUN_DIR / "dataset_manifest.json", dataset_manifest)
dataset_manifest

## 5. Copy and verify the frozen NVARC checkpoint

The copy is resumable at file granularity. The preflight checks the published shard sizes, Qwen3 architecture, tokenizer IDs, 16-row embeddings/output head, and BF16 weight dtype before vLLM starts.

In [ ]:
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer

required_model_files = (
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    *EXPECTED_NVARC_FILE_SIZES,
)
for relative_name in required_model_files:
    assert (MODEL_DRIVE_DIR / relative_name).is_file(), relative_name
for filename, expected_size in EXPECTED_NVARC_FILE_SIZES.items():
    assert (MODEL_DRIVE_DIR / filename).stat().st_size == expected_size, filename

source_size = sum(path.stat().st_size for path in MODEL_DRIVE_DIR.rglob("*") if path.is_file())
free_local_bytes = shutil.disk_usage("/content").free
MODEL_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
missing_bytes = sum(
    source.stat().st_size
    for source in MODEL_DRIVE_DIR.rglob("*")
    if source.is_file()
    and (
        not (MODEL_LOCAL_DIR / source.relative_to(MODEL_DRIVE_DIR)).is_file()
        or (MODEL_LOCAL_DIR / source.relative_to(MODEL_DRIVE_DIR)).stat().st_size != source.stat().st_size
    )
)
assert free_local_bytes >= missing_bytes + 2 * 10**9, (free_local_bytes, missing_bytes)

copy_started = time.perf_counter()
for source in sorted(MODEL_DRIVE_DIR.rglob("*")):
    relative = source.relative_to(MODEL_DRIVE_DIR)
    destination = MODEL_LOCAL_DIR / relative
    if source.is_dir():
        destination.mkdir(parents=True, exist_ok=True)
    elif not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(destination.suffix + ".copying")
        shutil.copy2(source, temporary)
        os.replace(temporary, destination)
copy_seconds = time.perf_counter() - copy_started

for relative_name in required_model_files:
    assert (MODEL_LOCAL_DIR / relative_name).is_file(), relative_name
for filename, expected_size in EXPECTED_NVARC_FILE_SIZES.items():
    assert (MODEL_LOCAL_DIR / filename).stat().st_size == expected_size, filename

raw_model_config = json.loads((MODEL_LOCAL_DIR / "config.json").read_text(encoding="utf-8"))
model_config = AutoConfig.from_pretrained(MODEL_LOCAL_DIR, local_files_only=True, trust_remote_code=False)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_LOCAL_DIR, local_files_only=True, trust_remote_code=False, use_fast=True
)
assert model_config.model_type == "qwen3", model_config.model_type
assert int(model_config.vocab_size) == 16, model_config.vocab_size
assert len(tokenizer) == 16, len(tokenizer)
for expected_id in range(10):
    assert tokenizer.convert_tokens_to_ids(str(expected_id)) == expected_id
assert tokenizer.convert_tokens_to_ids("Ċ") == 10
assert tokenizer.convert_tokens_to_ids("<|im_end|>") == EOS_TOKEN_ID
assert tokenizer.eos_token_id == EOS_TOKEN_ID

weight_index = json.loads(
    (MODEL_LOCAL_DIR / "model.safetensors.index.json").read_text(encoding="utf-8")
)["weight_map"]
tensor_contract = {}
for tensor_name in ("model.embed_tokens.weight", "lm_head.weight"):
    assert tensor_name in weight_index, tensor_name
    shard_path = MODEL_LOCAL_DIR / weight_index[tensor_name]
    with safe_open(str(shard_path), framework="pt", device="cpu") as handle:
        tensor = handle.get_tensor(tensor_name)
    assert tuple(tensor.shape)[0] == 16, (tensor_name, tuple(tensor.shape))
    assert tensor.dtype == torch.bfloat16, (tensor_name, tensor.dtype)
    tensor_contract[tensor_name] = {"shape": list(tensor.shape), "dtype": str(tensor.dtype)}
    del tensor

model_contract = {
    "source_handle": NVARC_HANDLE,
    "drive_path": str(MODEL_DRIVE_DIR),
    "local_path": str(MODEL_LOCAL_DIR),
    "source_size_bytes": source_size,
    "copy_seconds": copy_seconds,
    "architecture": raw_model_config.get("architectures"),
    "model_type": model_config.model_type,
    "vocab_size": int(model_config.vocab_size),
    "eos_token_id": tokenizer.eos_token_id,
    "weight_shard_sizes": EXPECTED_NVARC_FILE_SIZES,
    "tensor_contract": tensor_contract,
}
atomic_write_json(RUN_DIR / "model_contract.json", model_contract)
model_contract

In [ ]:
for relative_name in required_model_files:
    source = MODEL_DRIVE_DIR / relative_name
    destination = MODEL_LOCAL_DIR / relative_name

    if (
        not destination.is_file()
        or destination.stat().st_size != source.stat().st_size
    ):
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".copying")
        shutil.copy2(source, temporary)
        os.replace(temporary, destination)
        print(f"Copied: {relative_name}")

print("Required model files are present locally.")

In [ ]:
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer

In [ ]:
assert "tokenizer" in globals(), "Die Modell-/Tokenizer-Zelle muss zuerst erfolgreich laufen."
print(type(tokenizer), len(tokenizer))

## 6. Build canonical, target-free prompts

Prompts are tokenized explicitly with `add_special_tokens=False` and then passed to vLLM as token IDs. The notebook fails rather than dropping demonstrations if any complete prompt cannot leave room for a maximum 30×30 output.

In [ ]:
largest_reply = serialize_grid([[0] * 30 for _ in range(30)]) + "<|im_end|>"
MAX_NEW_TOKENS = len(tokenizer.encode(largest_reply, add_special_tokens=False)) + 1
assert MAX_NEW_TOKENS < MAX_MODEL_LEN

prepared_prompts: list[dict] = []
for task_id in selected_task_ids:
    task = evaluation_challenges[task_id]
    for test_index in range(len(task["test"])):
        prompt = format_nvarc_prompt(task, test_index)
        input_ids = tokenizer.encode(prompt, add_special_tokens=False)
        if len(input_ids) + MAX_NEW_TOKENS > MAX_MODEL_LEN:
            raise ValueError(
                f"{task_id}[{test_index}] requires {len(input_ids)} prompt tokens plus "
                f"{MAX_NEW_TOKENS} output tokens, exceeding {MAX_MODEL_LEN}"
            )
        prepared_prompts.append(
            {
                "task_id": task_id,
                "test_index": test_index,
                "input_ids": input_ids,
                "prompt_tokens": len(input_ids),
                "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
            }
        )

prompt_manifest = [
    {key: record[key] for key in ("task_id", "test_index", "prompt_tokens", "prompt_sha256")}
    for record in prepared_prompts
]
protocol = {
    "canonical_input_only": True,
    "direct_ancestral_sampling": True,
    "attempts_per_output": SAMPLES_PER_PROMPT,
    "test_time_training": False,
    "test_time_augmentation": False,
    "dfs_or_beam_search": False,
    "candidate_selection": False,
    "rescoring_or_ranking": False,
    "voting": False,
    "invalid_sample_retry": False,
    "target_shape_available_to_decoder": False,
    "strict_multi_output_attempt_alignment": True,
}
assert not any(
    protocol[name]
    for name in (
        "test_time_training",
        "test_time_augmentation",
        "dfs_or_beam_search",
        "candidate_selection",
        "rescoring_or_ranking",
        "voting",
        "invalid_sample_retry",
        "target_shape_available_to_decoder",
    )
)

config_identity = {
    "procedure_version": NOTEBOOK_PROCEDURE_VERSION,
    "run_name": RUN_NAME,
    "dataset_release": DATASET_RELEASE,
    "dataset_tasks_sha256": dataset_manifest["task_files_sha256"],
    "selected_task_ids": selected_task_ids,
    "excluded_context_task_ids": list(EXCLUDED_CONTEXT_TASK_IDS),
    "model_handle": NVARC_HANDLE,
    "model_weight_shard_sizes": EXPECTED_NVARC_FILE_SIZES,
    "vllm_version": VLLM_VERSION,
    "seed": SEED,
    "seed_namespace": SEED_NAMESPACE,
    "samples_per_prompt": SAMPLES_PER_PROMPT,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "max_model_len": MAX_MODEL_LEN,
    "max_new_tokens": MAX_NEW_TOKENS,
    "arc_token_ids": list(ARC_TOKEN_IDS),
    "protocol": protocol,
    "prompt_manifest_sha256": canonical_json_sha256(prompt_manifest),
}
CONFIG_FINGERPRINT = canonical_json_sha256(config_identity)
run_config = {
    **config_identity,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "runtime_preflight": runtime_preflight,
    "model_contract": model_contract,
    "dataset_manifest": dataset_manifest,
    "prompt_count": len(prepared_prompts),
    "prompt_batch_size": PROMPT_BATCH_SIZE,
    "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
}
run_config_path = RUN_DIR / "run_config.json"
if run_config_path.is_file():
    existing_config = json.loads(run_config_path.read_text(encoding="utf-8"))
    assert existing_config["config_fingerprint"] == CONFIG_FINGERPRINT, (
        existing_config["config_fingerprint"], CONFIG_FINGERPRINT
    )
else:
    atomic_write_json(run_config_path, run_config)

{
    "tasks": len(selected_task_ids),
    "prompts": len(prepared_prompts),
    "max_prompt_tokens": max(record["prompt_tokens"] for record in prepared_prompts),
    "max_new_tokens": MAX_NEW_TOKENS,
    "config_fingerprint": CONFIG_FINGERPRINT,
}

## 7. Load NVARC with vLLM

vLLM owns inference-only model execution and KV-cache scheduling. No optimizer, adapter, gradient-bearing path, or target data is passed to the engine.

In [ ]:
from vllm import LLM, SamplingParams

engine_started = time.perf_counter()
llm = LLM(
    model=str(MODEL_LOCAL_DIR),
    tokenizer=str(MODEL_LOCAL_DIR),
    dtype="bfloat16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    tensor_parallel_size=1,
    trust_remote_code=False,
    enable_prefix_caching=True,
)
engine_load_seconds = time.perf_counter() - engine_started
session_manifest = {
    "session_id": SESSION_ID,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "runtime": runtime_preflight,
    "engine_load_seconds": engine_load_seconds,
    "model_local_path": str(MODEL_LOCAL_DIR),
}
atomic_write_json(SESSION_DIR / f"{SESSION_ID}.json", session_manifest)
print(f"Loaded frozen NVARC in {engine_load_seconds:.1f}s.")

## 8. Batched, resumable pass@32 generation

Each prompt receives one deterministic vLLM request with 32 stochastic outputs. Seeds depend only on the protocol namespace, global seed, task ID, and test index—not batch composition or resume order. The parser accepts only rectangular 1×1 through 30×30 grids terminated by EOS.

In [ ]:
def result_path(record: dict) -> Path:
    return RESULT_DIR / f"{record['task_id']}__q{record['test_index']:03d}.json"


def valid_completed_result(path: Path, record: dict) -> bool:
    if not path.is_file():
        return False
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        payload.get("config_fingerprint") == CONFIG_FINGERPRINT
        and payload.get("task_id") == record["task_id"]
        and payload.get("test_index") == record["test_index"]
        and len(payload.get("samples", [])) == SAMPLES_PER_PROMPT
        and [sample.get("sample_index") for sample in payload["samples"]]
        == list(range(SAMPLES_PER_PROMPT))
    )


def json_safe_stop_reason(value: object) -> int | str | None:
    if value is None or isinstance(value, (int, str)):
        return value
    return str(value)


def completion_record(completion: object) -> dict:
    token_ids = [int(token_id) for token_id in completion.token_ids]
    assert all(token_id in ARC_TOKEN_IDS for token_id in token_ids), set(token_ids)
    finish_reason = getattr(completion, "finish_reason", None)
    stop_reason = json_safe_stop_reason(getattr(completion, "stop_reason", None))
    ended_with_eos = (
        bool(token_ids and token_ids[-1] == EOS_TOKEN_ID)
        or stop_reason == EOS_TOKEN_ID
        or (finish_reason == "stop" and stop_reason is None)
    )
    payload_ids = token_ids[:-1] if token_ids and token_ids[-1] == EOS_TOKEN_ID else token_ids
    decoded = tokenizer.decode(
        payload_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
    )
    grid = parse_grid_text(decoded) if ended_with_eos else None
    return {
        "sample_index": int(completion.index),
        "token_ids": token_ids,
        "generated_tokens": len(token_ids),
        "finish_reason": finish_reason,
        "stop_reason": stop_reason,
        "ended_with_eos": ended_with_eos,
        "valid": grid is not None,
        "grid": grid,
        "cumulative_logprob": getattr(completion, "cumulative_logprob", None),
    }


def sampling_params_for(record: dict) -> SamplingParams:
    return SamplingParams(
        n=SAMPLES_PER_PROMPT,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        min_p=0.0,
        presence_penalty=0.0,
        frequency_penalty=0.0,
        repetition_penalty=1.0,
        seed=stable_seed(
            SEED_NAMESPACE, SEED, record["task_id"], record["test_index"]
        ),
        max_tokens=MAX_NEW_TOKENS,
        stop_token_ids=[EOS_TOKEN_ID],
        allowed_token_ids=list(ARC_TOKEN_IDS),
        skip_special_tokens=False,
        spaces_between_special_tokens=False,
    )


def chunks(items: list[dict], size: int):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def generate_and_persist(records: list[dict], *, batch_size: int) -> dict:
    pending = [record for record in records if not valid_completed_result(result_path(record), record)]
    generated_prompts = 0
    skipped_prompts = len(records) - len(pending)
    for batch_number, batch in enumerate(chunks(pending, batch_size), start=1):
        batch_keys = [f"{record['task_id']}:{record['test_index']}" for record in batch]
        batch_id = (
            time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
            + "-"
            + canonical_json_sha256(batch_keys)[:12]
        )
        model_inputs = [{"prompt_token_ids": record["input_ids"]} for record in batch]
        params = [sampling_params_for(record) for record in batch]
        started = time.perf_counter()
        outputs = llm.generate(model_inputs, params, use_tqdm=True)
        elapsed_seconds = time.perf_counter() - started
        assert len(outputs) == len(batch)
        batch_generated_tokens = 0
        for record, output in zip(batch, outputs, strict=True):
            samples = sorted(
                (completion_record(completion) for completion in output.outputs),
                key=lambda sample: sample["sample_index"],
            )
            assert len(samples) == SAMPLES_PER_PROMPT, len(samples)
            assert [sample["sample_index"] for sample in samples] == list(range(SAMPLES_PER_PROMPT))
            batch_generated_tokens += sum(sample["generated_tokens"] for sample in samples)
            result = {
                "config_fingerprint": CONFIG_FINGERPRINT,
                "task_id": record["task_id"],
                "test_index": record["test_index"],
                "prompt_tokens": record["prompt_tokens"],
                "prompt_sha256": record["prompt_sha256"],
                "request_seed": stable_seed(
                    SEED_NAMESPACE, SEED, record["task_id"], record["test_index"]
                ),
                "batch_id": batch_id,
                "samples": samples,
            }
            atomic_write_json(result_path(record), result, indent=None)
            generated_prompts += 1
        event = {
            "config_fingerprint": CONFIG_FINGERPRINT,
            "batch_id": batch_id,
            "batch_number_in_session": batch_number,
            "prompt_keys": batch_keys,
            "prompt_count": len(batch),
            "sample_count": len(batch) * SAMPLES_PER_PROMPT,
            "generated_tokens": batch_generated_tokens,
            "elapsed_seconds": elapsed_seconds,
        }
        atomic_write_json(BATCH_EVENT_DIR / f"{batch_id}.json", event)
        print(
            f"Batch {batch_number}: {len(batch)} prompts, "
            f"{batch_generated_tokens} tokens, {elapsed_seconds:.1f}s"
        )
    return {
        "requested_prompts": len(records),
        "generated_prompts": generated_prompts,
        "skipped_completed_prompts": skipped_prompts,
    }

print(f"Ready to generate {len(prepared_prompts)} prompts × {SAMPLES_PER_PROMPT} samples.")

### Smoke batch

This writes the first two prompt results to the real run directory. The full-run cell below reuses them rather than sampling them again. A smoke output is not interpreted as a benchmark result.

In [ ]:
smoke_status = generate_and_persist(
    prepared_prompts[:SMOKE_PROMPTS], batch_size=SMOKE_PROMPTS
)
smoke_results = [
    json.loads(result_path(record).read_text(encoding="utf-8"))
    for record in prepared_prompts[:SMOKE_PROMPTS]
]
assert all(len(result["samples"]) == SAMPLES_PER_PROMPT for result in smoke_results)
smoke_summary = {
    **smoke_status,
    "samples": sum(len(result["samples"]) for result in smoke_results),
    "valid_samples": sum(
        int(sample["valid"]) for result in smoke_results for sample in result["samples"]
    ),
}
smoke_summary

### Full retained-subset run

This may take hours depending on GPU, output lengths, and Colab availability. Rerun the notebook after a disconnect; matching completed output files are skipped.

In [ ]:
full_run_status = generate_and_persist(prepared_prompts, batch_size=PROMPT_BATCH_SIZE)
missing_results = [
    f"{record['task_id']}:{record['test_index']}"
    for record in prepared_prompts
    if not valid_completed_result(result_path(record), record)
]
assert not missing_results, missing_results[:20]
full_run_status

## 9. Score exact grids after inference

Only this section reads `evaluation_targets`. For strict task pass@k, attempt `j` must solve every test output in the task at the same `j`. The diagnostic output-level metric instead asks whether each individual output appears anywhere among its first `k` samples.

In [ ]:
def score_pass_curve(
    task_ids: list[str],
    targets_by_task: dict[str, list[list[list[int]]]],
    results_by_output: dict[tuple[str, int], dict],
    pass_at_k: tuple[int, ...],
) -> tuple[list[dict], list[dict], list[dict]]:
    curve = []
    per_output = []
    per_task = []
    total_outputs = sum(len(targets_by_task[task_id]) for task_id in task_ids)

    correctness: dict[tuple[str, int], list[bool]] = {}
    for task_id in task_ids:
        for test_index, target in enumerate(targets_by_task[task_id]):
            result = results_by_output[(task_id, test_index)]
            samples = sorted(result["samples"], key=lambda sample: sample["sample_index"])
            assert len(samples) == max(pass_at_k)
            matches = [
                bool(sample["valid"] and exact_grid_match(sample["grid"], target))
                for sample in samples
            ]
            correctness[(task_id, test_index)] = matches
            valid_grids = [sample["grid"] for sample in samples if sample["valid"]]
            wrong_shape = sum(
                int(
                    len(grid) != len(target)
                    or len(grid[0]) != len(target[0])
                )
                for grid in valid_grids
            )
            first_correct_rank = next((index + 1 for index, match in enumerate(matches) if match), None)
            per_output.append(
                {
                    "task_id": task_id,
                    "test_index": test_index,
                    "valid_samples": len(valid_grids),
                    "invalid_samples": len(samples) - len(valid_grids),
                    "unique_valid_grids": len(
                        {tuple(tuple(row) for row in grid) for grid in valid_grids}
                    ),
                    "wrong_shape_valid_samples": wrong_shape,
                    "first_correct_rank": first_correct_rank,
                    "target_height": len(target),
                    "target_width": len(target[0]),
                }
            )

    for task_id in task_ids:
        output_count = len(targets_by_task[task_id])
        task_row = {"task_id": task_id, "output_count": output_count}
        for k in pass_at_k:
            task_row[f"strict_pass_at_{k}"] = any(
                all(correctness[(task_id, test_index)][attempt_index] for test_index in range(output_count))
                for attempt_index in range(k)
            )
        per_task.append(task_row)

    for k in pass_at_k:
        solved_outputs = sum(
            int(any(matches[:k])) for matches in correctness.values()
        )
        task_averaged_output_pass = sum(
            sum(
                int(any(correctness[(task_id, test_index)][:k]))
                for test_index in range(len(targets_by_task[task_id]))
            )
            / len(targets_by_task[task_id])
            for task_id in task_ids
        ) / len(task_ids)
        strict_solved_tasks = sum(int(row[f"strict_pass_at_{k}"]) for row in per_task)
        curve.append(
            {
                "k": k,
                "strict_task_pass_at_k": strict_solved_tasks / len(task_ids),
                "strict_solved_tasks": strict_solved_tasks,
                "task_count": len(task_ids),
                "output_weighted_pass_at_k": solved_outputs / total_outputs,
                "solved_outputs": solved_outputs,
                "output_count": total_outputs,
                "task_averaged_output_pass_at_k": task_averaged_output_pass,
            }
        )
    return curve, per_task, per_output


# Synthetic multi-output test: only aligned attempt 1 solves both outputs.
_synthetic_results = {
    ("x", 0): {"samples": [
        {"sample_index": 0, "valid": True, "grid": [[1]]},
        {"sample_index": 1, "valid": True, "grid": [[1]]},
    ]},
    ("x", 1): {"samples": [
        {"sample_index": 0, "valid": True, "grid": [[0]]},
        {"sample_index": 1, "valid": True, "grid": [[2]]},
    ]},
}
_curve, _, _ = score_pass_curve(["x"], {"x": [[[1]], [[2]]]}, _synthetic_results, (1, 2))
assert _curve[0]["strict_solved_tasks"] == 0
assert _curve[0]["output_weighted_pass_at_k"] == 0.5
assert _curve[1]["strict_solved_tasks"] == 1
assert _curve[1]["output_weighted_pass_at_k"] == 1.0
print("Strict multi-output scoring self-test passed.")

In [ ]:
import csv

import pandas as pd

results_by_output = {}
for record in prepared_prompts:
    path = result_path(record)
    assert valid_completed_result(path, record), path
    payload = json.loads(path.read_text(encoding="utf-8"))
    results_by_output[(record["task_id"], record["test_index"])] = payload

pass_curve, per_task_rows, per_output_rows = score_pass_curve(
    selected_task_ids, evaluation_targets, results_by_output, PASS_AT_K
)
all_samples = [
    sample
    for result in results_by_output.values()
    for sample in result["samples"]
]
batch_events = []
for event_path in sorted(BATCH_EVENT_DIR.glob("*.json")):
    event = json.loads(event_path.read_text(encoding="utf-8"))
    if event.get("config_fingerprint") == CONFIG_FINGERPRINT:
        batch_events.append(event)
total_batch_seconds = sum(event["elapsed_seconds"] for event in batch_events)
total_generated_tokens = sum(sample["generated_tokens"] for sample in all_samples)
diagnostics = {
    "generated_samples": len(all_samples),
    "valid_samples": sum(int(sample["valid"]) for sample in all_samples),
    "invalid_samples": sum(int(not sample["valid"]) for sample in all_samples),
    "eos_terminated_samples": sum(int(sample["ended_with_eos"]) for sample in all_samples),
    "generated_tokens": total_generated_tokens,
    "recorded_batch_seconds": total_batch_seconds,
    "generated_tokens_per_second": (
        total_generated_tokens / total_batch_seconds if total_batch_seconds else None
    ),
    "engine_load_seconds": engine_load_seconds,
    "invalid_samples_count_as_incorrect": True,
    "wrong_shape_samples_were_filtered": False,
}
metrics = {
    "config_fingerprint": CONFIG_FINGERPRINT,
    "contaminated_public_evaluation": True,
    "excluded_task_ids": list(EXCLUDED_CONTEXT_TASK_IDS),
    "interpretation": (
        "Released-checkpoint behavior on an explicitly filtered 396/400 public subset; "
        "not a full-set score or a generalization estimate."
    ),
    "primary_metric": "strict_task_pass_at_k",
    "pass_curve": pass_curve,
    "diagnostics": diagnostics,
    "protocol": protocol,
}
atomic_write_json(RUN_DIR / "metrics.json", metrics)

def write_csv(path: Path, rows: list[dict]) -> None:
    assert rows
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)

write_csv(RUN_DIR / "pass_curve.csv", pass_curve)
write_csv(RUN_DIR / "per_task.csv", per_task_rows)
write_csv(RUN_DIR / "per_output.csv", per_output_rows)

summary = {
    "run_dir": str(RUN_DIR),
    "tasks": len(selected_task_ids),
    "outputs": len(prepared_prompts),
    "excluded_task_ids": list(EXCLUDED_CONTEXT_TASK_IDS),
    "strict_task_pass_at_32": pass_curve[-1]["strict_task_pass_at_k"],
    "strict_solved_tasks_at_32": pass_curve[-1]["strict_solved_tasks"],
    "valid_sample_rate": diagnostics["valid_samples"] / diagnostics["generated_samples"],
}
atomic_write_json(RUN_DIR / "run_summary.json", summary)
display(pd.DataFrame(pass_curve))
summary

## Produced artifacts and interpretation

The deterministic Drive run directory contains:

- `run_config.json`, `dataset_manifest.json`, and `model_contract.json` for reproducibility;
- `output_results/*.json` with all token IDs, parsed grids, validity flags, seeds, and finish reasons;
- `batch_events/*.json` with measured vLLM throughput;
- `metrics.json`, `pass_curve.csv`, `per_task.csv`, `per_output.csv`, and `run_summary.json`.

Report strict task pass@32 explicitly as a **396/400 subset result**, list the four excluded IDs, and keep the contamination caveat attached. It is not a full ARC-AGI-1 evaluation score. Do not compare it directly with hidden-set or TTT-enabled numbers as though the protocol and data exposure were controlled. Stable request seeds make resumed runs reproducible within the pinned setup, but byte-identical samples across different GPUs or future CUDA kernels are not guaranteed.

## Next steps

- Preserve the completed Drive run directory before changing any sampling setting.
- If another temperature is tested, use a new deterministic run name and treat it as a separate experiment.
- A useful later comparison is the same pass@32 protocol on a genuinely held-out task set; do not tune on this contaminated public score.